In [3]:
from pathlib import Path
import itertools
import subprocess
import sys
from datetime import datetime

project_dir = Path("/scratch/rhm4nj/gpu_arch/gsplat-fork")
# project_dir = Path("/bigtemp/rhm4nj/gpu_arch/project/gsplat-fork")
TEMPLATE_PATH = str(project_dir/ "scripts/slurm/profile_gsplat_template_rivanna.slurm")
mode = "grid"   # grid or zip - grid does cartesian product of all params, zip pairs them by index (so all lists must be same length)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
base_dir = "outputs" / Path(timestamp)
slurm_dir = base_dir / "slurm_scripts"

slurm_out_dir = base_dir / "slurm_out"
slurm_err_dir = base_dir / "slurm_err"
config_dir = base_dir / "configs"
results_dir = project_dir / "scripts/slurm" / base_dir / "results"

slurm_dir.mkdir(parents=True, exist_ok=False)
slurm_out_dir.mkdir(parents=True, exist_ok=False)
slurm_err_dir.mkdir(parents=True, exist_ok=False)
config_dir.mkdir(parents=True, exist_ok=False)
results_dir.mkdir(parents=True, exist_ok=False)

fixed_params = {
    "script_path": project_dir / "examples/benchmarks/custom_mod.sh",
    "partition": "gpu",
    "log_out": slurm_out_dir / "%x-%j.out",
    "log_err": slurm_err_dir / "%x-%j.err",
    "module_loads": "module load cuda\nmodule load nsight-systems",
    "trace": "cuda,osrt,nvtx",
    "sample": "cpu",
    # "load_env_cmd": "source /bigtemp/rhm4nj/gpu_arch/project/gsplat-env/bin/activate",
    "project_dir": project_dir,
    "gpus": 1,
    "cpus": 8,
    "mem": "32G",
    "time": "02:00:00",
    "force_overwrite": "true",
    "gpu_type": "a6000",
    "scenes": "garden",
    # profile_output and extra_trainer_args are set per-job automatically
}

# scenes = ["garden", "bicycle", "stump", "bonsai", "counter", "kitchen", "room"]

sweep_params = {
    # "scenes": ['garden'],
    "--optimizer-stride": [1, 4],
    "--enable-frustum-culling": [True, False],
    "--enable-input-cache": [True, False],
}

# --- Profile output naming ---
profile_prefix = str(results_dir / "profile_gsplat")

def _abbrev(arg: str) -> str:
    """First letter of each word in a --kebab-case arg. e.g. --enable-frustum-culling -> efc"""
    return "".join(w[0] for w in arg.lstrip("-").split("-"))

def make_profile_output(prefix: str, keys: list, combo: tuple) -> str:
    """Build a name like: profile_gsplat_os-1_efc-True_eic-False"""
    parts = [f"{_abbrev(k)}-{v}" for k, v in zip(keys, combo)]
    return "_".join([prefix] + parts)

def make_extra_trainer_args(keys: list, combo: tuple) -> str:
    """Convert sweep params to shell flags forwarded to trainer.py.
    - bool True  -> --flag-name
    - bool False -> --no-flag-name  (tyro convention)
    - other      -> --flag-name value
    """
    parts = []
    for k, v in zip(keys, combo):
        if isinstance(v, bool):
            parts.append(k if v else "--no-" + k.lstrip("-"))
        else:
            parts.append(f"{k} {v}")
    return " ".join(parts)

keys = list(sweep_params.keys())
values = list(sweep_params.values())

if mode == "grid": combinations = list(itertools.product(*values))

elif mode == "zip":
    lengths = [len(v) for v in values]
    if len(set(lengths)) != 1:
        raise ValueError("all sweep parameter lists must have same length in zip mode.")
    combinations = list(zip(*values))

else:
    raise ValueError("mode has to be 'grid' or 'zip'")

total_jobs = len(combinations)
print(f"\ntotal jobs to generate: {total_jobs}")
print("\nJobs:")
for combo in combinations:
    print(f"  {make_profile_output(profile_prefix, keys, combo)}")
    print(f"    args: {make_extra_trainer_args(keys, combo)}")


total jobs to generate: 8

Jobs:
  /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-True_eic-True
    args: --optimizer-stride 1 --enable-frustum-culling --enable-input-cache
  /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-True_eic-False
    args: --optimizer-stride 1 --enable-frustum-culling --no-enable-input-cache
  /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-False_eic-True
    args: --optimizer-stride 1 --no-enable-frustum-culling --enable-input-cache
  /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-False_eic-False
    args: --optimizer-stride 1 --no-enable-frustum-culling --no-enable-input-cache
  /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-4_efc-True_eic-True
    arg

In [4]:
from pathlib import Path

folders = [Path(slurm_dir), Path(config_dir)]
for folder in folders:
    if any(folder.iterdir()):
        raise RuntimeError("Output folder already has outputs - re-run above cell")

do_run = False
confirm = input("do you want to continue? (y/n): ")
do_run = confirm.lower() == "y"

print("generating and submitting jobs...\n")

template = Path(TEMPLATE_PATH).read_text()
if "gpu_type" not in sweep_params and "gpu_type" not in fixed_params:
    template = template.replace(r'#SBATCH --constraint="{gpu_type}"', "")

for idx, combo in enumerate(combinations):
    job_params = dict(fixed_params)
    job_params["profile_output"] = make_profile_output(profile_prefix, keys, combo)
    job_params["extra_trainer_args"] = make_extra_trainer_args(keys, combo)
    filled = template.format(**job_params)
    job_file = slurm_dir / f"job_{idx}.slurm"
    job_file.write_text(filled)
    cfg_file = config_dir / f"job_{idx}.cfg"

    with open(cfg_file, "w") as f:
        for k, v in job_params.items():
            f.write(f"{k}={v}\n")

    print(f"  job_{idx}: {job_params['profile_output']}")
    print(f"    {job_params['extra_trainer_args']}")
    if do_run:
        subprocess.run(["sbatch", str(job_file)])
        print("  Submitted:", str(job_file))

print("\nAll jobs generated successfully.")

generating and submitting jobs...

  job_0: /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-True_eic-True
    --optimizer-stride 1 --enable-frustum-culling --enable-input-cache
Submitted batch job 10393553
  Submitted: outputs/2026-03-21_16-31-29/slurm_scripts/job_0.slurm
  job_1: /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-True_eic-False
    --optimizer-stride 1 --enable-frustum-culling --no-enable-input-cache
Submitted batch job 10393554
  Submitted: outputs/2026-03-21_16-31-29/slurm_scripts/job_1.slurm
  job_2: /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/slurm/outputs/2026-03-21_16-31-29/results/profile_gsplat_os-1_efc-False_eic-True
    --optimizer-stride 1 --no-enable-frustum-culling --enable-input-cache
Submitted batch job 10393555
  Submitted: outputs/2026-03-21_16-31-29/slurm_scripts/job_2.slurm
  job_3: /scratch/rhm4nj/gpu_arch/gsplat-fork/scripts/